# index_analysis — 시장지수 자체 분석

- 담당: (이름)  브랜치: `<본인 github id>`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.
- 선행: `python -m backend.services.market.collector` 로 `data/market/index.csv` 생성

In [ ]:
import pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
mi = pd.read_csv(ROOT/"data/market/index.csv")
mi.columns = mi.columns.str.strip().str.lower()
mi["trade_date"] = pd.to_datetime(mi["trade_date"])
mi.info(); mi.groupby("index_name")["trade_date"].agg(["min","max","count"])

## 1. 지수별 평균·최대·최소·변동성

In [ ]:
piv = mi.pivot(index="trade_date", columns="index_name", values="close_value").sort_index()
ret = (piv.pct_change()*100).round(2)
summary = pd.DataFrame({"mean":piv.mean(), "max":piv.max(), "min":piv.min(),
                        "max_date":piv.idxmax(), "min_date":piv.idxmin(),
                        "daily_vol":ret.std().round(3)})
summary

## 2. 변화가 큰 구간 (일간 상위 5, 5일 이동 변화율)

In [ ]:
top_moves = ret.abs().stack().nlargest(5).reset_index()
top_moves.columns = ["trade_date","index_name","abs_pct"]
roll5 = (piv.pct_change(5)*100).round(2)
top_moves, roll5.abs().stack().nlargest(5)

## 3. 결과 내보내기

In [ ]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
summary.reset_index().to_csv(out/"pandas_E_index_summary.csv", index=False)
ret.reset_index().to_csv(out/"pandas_E_index_returns.csv", index=False)